# Clean Hourly Beijing Data, Create Daily Pollutants, Join ERA5, and Extend Through 2025

This notebook cleans the old 2013–2017 Beijing hourly source, converts pollutant readings into daily station records, and joins those records to the ERA5 daily weather features by station ID and date.

It writes two separate outputs in `air-quality-classification/data/`:

- `daily_pollutant_readings_2013-2017.csv` — daily pollutant values and valid-hour counts.
- `daily_pollutant_readings_with_era5_weather_2013-2017.csv` — the daily pollutant CSV joined to ERA5 daily features.

The shared AQI helper applies 24-hour means with at least 20 valid readings for PM2.5, PM10, SO2, NO2, and CO; O3 uses the maximum trailing 8-hour mean for the day. CO is in mg/m³ and the other pollutant values are in µg/m³. ERA5 values are reanalysis estimates, not direct monitor measurements.

## 1. Locate files and shared aggregation rules

Find the repository root, locate the original hourly data and ERA5 daily features, and import the same pollutant aggregation helper used in the model workflow.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import os
from sqlalchemy import create_engine

REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "air-quality-classification" / "data" / "cleaned_dataset.csv").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the repository and cleaned hourly CSV.")

PROJECT_DIR = REPO_ROOT / "air-quality-classification"
SOURCE_FILE = PROJECT_DIR / "data" / "cleaned_dataset.csv"
POLLUTANT_OUTPUT = PROJECT_DIR / "data" / "daily_pollutant_readings_2013-2017.csv"
ERA5_DAILY_FILE = REPO_ROOT / "CNEMC_Figshare_2014-2025" / "data" / "beijing_station_weather_era5_daily_features.csv"
COMBINED_OUTPUT = PROJECT_DIR / "data" / "daily_pollutant_readings_with_era5_weather_2013-2017.csv"
NEWER_DAILY_WEATHER_FILE = REPO_ROOT / "CNEMC_Figshare_2014-2025" / "data" / "beijing_project_stations_daily_with_weather.csv"
EXTENDED_OUTPUT = PROJECT_DIR / "data" / "daily_pollutant_readings_with_era5_weather_2013-2025.csv"
DB_URL = os.environ.get("AIR_QUALITY_DB_URL", "mysql+pymysql://root:@localhost/air_quality")
DATABASE_TABLE = "aqi_daily"
HELPER_DIR = PROJECT_DIR / "app"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))
from aqi_labels import build_daily_labels, require_columns

for path in [SOURCE_FILE, ERA5_DAILY_FILE, NEWER_DAILY_WEATHER_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input is missing: {path}")
print(f"Hourly source: {SOURCE_FILE}")
print(f"ERA5 daily source: {ERA5_DAILY_FILE}")
print(f"Newer daily pollutant + ERA5 source: {NEWER_DAILY_WEATHER_FILE}")

Hourly source: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data\cleaned_dataset.csv
ERA5 daily source: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\CNEMC_Figshare_2014-2025\data\beijing_station_weather_era5_daily_features.csv
Newer daily pollutant + ERA5 source: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\CNEMC_Figshare_2014-2025\data\beijing_project_stations_daily_with_weather.csv


## 2. Load the hourly readings

Load the 418k-row hourly CSV and check that pollutant and weather source columns are present. This cell reports its station and date coverage before cleaning.

In [2]:
source_df = pd.read_csv(SOURCE_FILE, low_memory=False)
require_columns(source_df, "hourly Beijing source")
HOURLY_WEATHER = ["TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM"]
missing = [c for c in HOURLY_WEATHER if c not in source_df.columns]
if missing:
    raise ValueError(f"Hourly source lacks weather fields: {missing}")
source_dates = pd.to_datetime(source_df[["year", "month", "day"]], errors="coerce")
print(f"Rows: {len(source_df):,}; stations: {source_df['station'].nunique()}")
print(f"Dates: {source_dates.min().date()} to {source_dates.max().date()}")

Rows: 418,753; stations: 12
Dates: 2013-03-01 to 2017-02-28


## 3. Clean invalid identities, duplicates, and readings

Follow the project cleaning rules: remove invalid station/date identities, collapse exact repeated station-hour rows, stop if one station-hour has conflicting readings, coerce numeric fields, and mark impossible values as missing. Unknown wind directions are also set to missing. The source file is not overwritten and missing readings are not imputed.

In [3]:
EXPECTED_STATIONS = {
    "aotizhongxin", "changping", "dingling", "dongsi", "guanyuan", "gucheng",
    "huairou", "nongzhanguan", "shunyi", "tiantan", "wanliu", "wanshouxigong",
}
TIME_COLUMNS = ["year", "month", "day", "hour"]
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
READINGS = [*POLLUTANTS, *HOURLY_WEATHER]
parts = source_df[TIME_COLUMNS].apply(pd.to_numeric, errors="coerce")
integer_parts = parts.notna() & parts.mod(1).eq(0)
safe_parts = parts.fillna(-1).astype("int64")
timestamps = pd.to_datetime({
    "year": safe_parts.year, "month": safe_parts.month,
    "day": safe_parts.day, "hour": safe_parts.hour,
}, errors="coerce")
valid_time = timestamps.notna() & integer_parts.all(axis=1)
station_key = source_df.station.astype("string").str.strip().str.casefold()
valid_station = station_key.isin(EXPECTED_STATIONS)
invalid_identity = ~(valid_time & valid_station)
work = source_df.loc[~invalid_identity].copy()
work["_timestamp"] = timestamps.loc[~invalid_identity]
work["_station_key"] = station_key.loc[~invalid_identity]
key = ["_station_key", "_timestamp"]
repeated = work.loc[work.duplicated(key, keep=False)]
if repeated.empty:
    conflicts = pd.Series(dtype="int64")
else:
    distinct = repeated.groupby(key, dropna=False)[READINGS].apply(lambda x: len(x.drop_duplicates()))
    conflicts = distinct[distinct > 1]
if not conflicts.empty:
    raise RuntimeError(f"{len(conflicts):,} station-hour keys have conflicting readings; review them before continuing.")
rows_before = len(work)
work = work.drop_duplicates(subset=[*key, *READINGS], keep="first").copy()
duplicates_removed = rows_before - len(work)

NONNEGATIVE = [*POLLUTANTS, "RAIN", "WSPM"]
NUMERIC = [*POLLUTANTS, "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
invalid_cells = {}
for column in NUMERIC:
    raw = work[column]
    values = pd.to_numeric(raw, errors="coerce")
    invalid = raw.notna() & values.isna()
    if column in NONNEGATIVE:
        invalid |= values.lt(0)
    if column == "PRES":
        invalid |= values.le(0)
    invalid_cells[column] = int(invalid.sum())
    work[column] = values.mask(invalid)

COMPASS = {"N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"}
wd = work.wd.astype("string").str.strip().str.upper()
invalid_wd = wd.notna() & ~wd.isin(COMPASS)
invalid_cells["wd"] = int(invalid_wd.sum())
work["wd"] = wd.mask(invalid_wd)
for column in TIME_COLUMNS:
    work[column] = parts.loc[work.index, column].astype("int64")
cleaned_hourly = work.drop(columns=["_timestamp", "_station_key"]).copy()
if "interpolated_columns" not in cleaned_hourly:
    cleaned_hourly["interpolated_columns"] = "None"

print(f"Invalid station/date rows removed: {int(invalid_identity.sum()):,}")
print(f"Exact duplicate station-hour rows removed: {duplicates_removed:,}")
print(f"Cleaned hourly rows retained: {len(cleaned_hourly):,}")
print("Invalid reading cells set to missing:", invalid_cells)

Invalid station/date rows removed: 0
Exact duplicate station-hour rows removed: 0
Cleaned hourly rows retained: 418,753
Invalid reading cells set to missing: {'PM2.5': 0, 'PM10': 0, 'SO2': 0, 'NO2': 0, 'CO': 0, 'O3': 0, 'TEMP': 0, 'PRES': 0, 'DEWP': 0, 'RAIN': 0, 'WSPM': 0, 'wd': 0}


## 4. Create the daily pollutant CSV

Use the project's shared daily AQI helper to calculate the pollutant summaries, then save just the daily pollutant values and valid-hour counts. This output has no AQI target labels.

In [4]:
daily_records, hourly_rows_with_labels = build_daily_labels(cleaned_hourly)
del hourly_rows_with_labels
pollutant_names = {
    "PM2.5_24h_avg": "PM2.5", "PM10_24h_avg": "PM10",
    "SO2_24h_avg": "SO2", "NO2_24h_avg": "NO2",
    "CO_mg_m3": "CO", "O3_8h_avg": "O3",
}
quality_columns = ["PM2.5_valid_hours", "PM10_valid_hours", "SO2_valid_hours",
                   "NO2_valid_hours", "CO_valid_hours", "O3_1h_max"]
selected = ["date", "station", *pollutant_names, *quality_columns]
missing = [c for c in selected if c not in daily_records.columns]
if missing:
    raise ValueError(f"Daily helper output lacks columns: {missing}")
daily_pollutants = daily_records[selected].rename(columns=pollutant_names).copy()
daily_pollutants["date"] = pd.to_datetime(daily_pollutants.date, errors="coerce").dt.strftime("%Y-%m-%d")
daily_pollutants = daily_pollutants.sort_values(["date", "station"]).reset_index(drop=True)
if daily_pollutants.duplicated(["date", "station"]).any():
    raise ValueError("Duplicate station-days found in daily pollutant output.")
POLLUTANT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
daily_pollutants.to_csv(POLLUTANT_OUTPUT, index=False, encoding="utf-8")
print(f"Daily pollutant station-days: {len(daily_pollutants):,}")
print(f"Saved: {POLLUTANT_OUTPUT}")

Daily pollutant station-days: 17,492
Saved: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data\daily_pollutant_readings_2013-2017.csv


## 5. Validate the ERA5 daily table and map station IDs

Read the prepared ERA5 daily features, validate its date/station keys and numeric fields, and check basic physical ranges. A station-name crosswalk assigns ERA5 station IDs to the older pollutant-only daily CSV, which has station names but no IDs.

In [5]:
era5 = pd.read_csv(ERA5_DAILY_FILE, dtype={"station_id": "string"}, low_memory=False)
WEATHER_FEATURES = ["TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
                    "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos"]
WEATHER_COUNTS = ["temperature_hours", "pressure_hours", "dewpoint_hours", "humidity_hours",
                  "rain_hours", "wind_speed_hours", "wind_direction_hours"]
required = ["date", "station_id", "station", *WEATHER_FEATURES, *WEATHER_COUNTS]
missing = [c for c in required if c not in era5.columns]
if missing:
    raise ValueError(f"ERA5 daily file lacks columns: {missing}")
era5["date"] = pd.to_datetime(era5.date, errors="coerce").dt.strftime("%Y-%m-%d")
era5["station_id"] = era5.station_id.astype("string").str.strip()
era5["station"] = era5.station.astype("string").str.strip()
if era5[["date", "station_id", "station"]].isna().any().any():
    raise ValueError("ERA5 data has missing or invalid date/station keys.")
if era5.duplicated(["station_id", "date"]).any():
    raise ValueError("ERA5 data contains duplicate station-days.")

weather_cleaning_counts = {}
for column in [*WEATHER_FEATURES, *WEATHER_COUNTS]:
    raw = era5[column]
    values = pd.to_numeric(raw, errors="coerce")
    invalid = raw.notna() & values.isna()
    if column == "PRES_mean": invalid |= values.le(0)
    if column in ["RAIN_sum", "WSPM_mean"]: invalid |= values.lt(0)
    if column == "RH_mean": invalid |= ~values.between(0, 100)
    if column in ["wd_sin", "wd_cos"]: invalid |= ~values.between(-1, 1)
    if column in WEATHER_COUNTS: invalid |= ~values.between(0, 24) | values.mod(1).ne(0)
    weather_cleaning_counts[column] = int(invalid.sum())
    era5[column] = values.mask(invalid)

# Each station name and ID must map to exactly one another.
era5["_station_key"] = era5.station.astype("string").str.strip().str.casefold()
station_map = era5[["station_id", "_station_key"]].drop_duplicates()
if station_map.station_id.duplicated().any() or station_map._station_key.duplicated().any():
    raise ValueError("ERA5 station name to ID crosswalk is not one-to-one.")
print(f"ERA5 daily station-days: {len(era5):,}; stations: {era5.station_id.nunique()}")
print("ERA5 invalid cells set to missing:", weather_cleaning_counts)

ERA5 daily station-days: 51,936; stations: 12
ERA5 invalid cells set to missing: {'TEMP_mean': 0, 'TEMP_range': 0, 'PRES_mean': 0, 'DEWP_mean': 0, 'RH_mean': 0, 'RAIN_sum': 0, 'WSPM_mean': 0, 'wd_sin': 0, 'wd_cos': 0, 'temperature_hours': 0, 'pressure_hours': 0, 'dewpoint_hours': 0, 'humidity_hours': 0, 'rain_hours': 0, 'wind_speed_hours': 0, 'wind_direction_hours': 0}


## 6. Merge the newly created pollutants CSV with ERA5

Read the pollutant CSV just written, map its station names to ERA5 station IDs, and join on both station ID and date. The left join keeps every pollutant station-day. Match flags make any missing weather rows or station-name mismatches visible.

In [6]:
daily_pollutants = pd.read_csv(POLLUTANT_OUTPUT, dtype={"station": "string"}, low_memory=False)
daily_pollutants["date"] = pd.to_datetime(daily_pollutants.date, errors="coerce").dt.strftime("%Y-%m-%d")
daily_pollutants["_station_key"] = daily_pollutants.station.astype("string").str.strip().str.casefold()
daily_pollutants = daily_pollutants.merge(station_map, on="_station_key", how="left", validate="many_to_one")
if daily_pollutants.station_id.isna().any():
    unknown = sorted(daily_pollutants.loc[daily_pollutants.station_id.isna(), "station"].dropna().unique())
    raise ValueError(f"No ERA5 station ID mapping for: {unknown}")

weather_for_join = era5.drop(columns=["_station_key"]).rename(columns={"station": "weather_station"})
combined_daily = daily_pollutants.merge(
    weather_for_join, on=["station_id", "date"], how="left", validate="one_to_one", indicator="_weather_join"
)
combined_daily["weather_matched"] = combined_daily["_weather_join"].eq("both")
combined_daily["weather_station_matches_pollutant"] = (
    combined_daily.station.astype("string").str.casefold()
    == combined_daily.weather_station.astype("string").str.casefold()
).fillna(False)
weather_matches = int(combined_daily.weather_matched.sum())
weather_match_percent = 100 * weather_matches / len(combined_daily) if len(combined_daily) else 0
combined_daily = combined_daily.drop(columns=["_station_key", "_weather_join"])
combined_daily = combined_daily.sort_values(["date", "station"]).reset_index(drop=True)
if combined_daily.duplicated(["station_id", "date"]).any():
    raise ValueError("Combined ERA5 data contains duplicate station-days.")
print(f"Combined station-days: {len(combined_daily):,}")
print(f"ERA5 weather matches: {weather_matches:,}/{len(combined_daily):,} ({weather_match_percent:.2f}%)")
print(f"Station name matches: {int(combined_daily.weather_station_matches_pollutant.sum()):,}/{len(combined_daily):,}")

Combined station-days: 17,492
ERA5 weather matches: 17,492/17,492 (100.00%)
Station name matches: 17,492/17,492


## 7. Save and review the combined daily CSV

Save the weather-inclusive dataset under a new filename, leaving the pollutant-only CSV and all original source files intact. The final summary shows date coverage, weather match coverage, row uniqueness, and the output columns.

In [7]:
COMBINED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
combined_daily.to_csv(COMBINED_OUTPUT, index=False, encoding="utf-8")
print(f"Saved: {COMBINED_OUTPUT}")
print(f"Rows: {len(combined_daily):,}; columns: {len(combined_daily.columns)}")
print(f"Dates: {combined_daily.date.min()} to {combined_daily.date.max()}")
print(f"Stations: {combined_daily.station.nunique()}; duplicate station-days: {int(combined_daily.duplicated(['station_id', 'date']).sum())}")
print(f"File size: {COMBINED_OUTPUT.stat().st_size / (1024 ** 2):.2f} MiB")
print(combined_daily.head(5).to_string(index=False))

Saved: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data\daily_pollutant_readings_with_era5_weather_2013-2017.csv
Rows: 17,492; columns: 34
Dates: 2013-03-01 to 2017-02-28
Stations: 12; duplicate station-days: 0
File size: 4.91 MiB
      date      station    PM2.5      PM10       SO2       NO2       CO     O3  PM2.5_valid_hours  PM10_valid_hours  SO2_valid_hours  NO2_valid_hours  CO_valid_hours  O3_1h_max station_id weather_station  TEMP_mean  TEMP_range   PRES_mean  DEWP_mean   RH_mean  RAIN_sum  WSPM_mean    wd_sin   wd_cos  temperature_hours  pressure_hours  dewpoint_hours  humidity_hours  rain_hours  wind_speed_hours  wind_direction_hours  weather_matched weather_station_matches_pollutant
2013-03-01 Aotizhongxin 7.125000 10.750000 11.708333 22.583333 0.429167 74.500                 24                24               24               24              24       77.0      1011A    Aotizhongxin   0.508333         9.0 1026.658333 -17.933333 24.291667   

## 8. Append the newer 2014–2025 pollutant-and-weather data

The newer file already contains daily pollutants and ERA5 features. Rename its pollutant columns to match the older table, select the exact same 34-column schema, then append it to the 2013–2017 result. For overlapping station-days, keep the older row; add only station-days that are not already present. The final CSV therefore spans the union of both periods without duplicate station-days.


In [8]:
# Read the newer daily file, which already includes pollutant and ERA5 weather features.
newer_daily = pd.read_csv(NEWER_DAILY_WEATHER_FILE, low_memory=False)

# Map the newer file's pollutant field names to the established daily schema.
newer_daily = newer_daily.rename(columns={
    "station_name": "station",
    "PM2.5_24h_avg": "PM2.5",
    "PM10_24h_avg": "PM10",
    "SO2_24h_avg": "SO2",
    "NO2_24h_avg": "NO2",
    "CO_24h_avg": "CO",
    "O3_daily_max_8h_avg": "O3",
    "O3_daily_max_1h": "O3_1h_max",
})

# Require the mapped fields, then use precisely the same columns and order as the older output.
final_columns = combined_daily.columns.tolist()
missing = sorted(set(final_columns) - set(newer_daily.columns))
if missing:
    raise ValueError(f"Newer data are missing required columns after renaming: {missing}")
newer_daily = newer_daily[final_columns].copy()

# Normalize join keys before comparing station-days.
for frame in (combined_daily, newer_daily):
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    frame["station_id"] = frame["station_id"].astype("string").str.strip()
if combined_daily[["date", "station_id"]].isna().any().any() or newer_daily[["date", "station_id"]].isna().any().any():
    raise ValueError("A source contains a missing or invalid station-day key.")
if newer_daily.duplicated(["station_id", "date"]).any():
    raise ValueError("The newer source contains duplicate station-days; resolve them before appending.")

old_rows = len(combined_daily)
new_rows = len(newer_daily)
old_keys = set(map(tuple, combined_daily[["station_id", "date"]].to_numpy()))
new_keys = set(map(tuple, newer_daily[["station_id", "date"]].to_numpy()))
overlap_count = len(old_keys & new_keys)
added_count = len(new_keys - old_keys)

# Older rows come first, so drop_duplicates keeps the existing values on overlap.
extended_daily = pd.concat([combined_daily, newer_daily], ignore_index=True)
extended_daily = extended_daily.drop_duplicates(["station_id", "date"], keep="first")
extended_daily = extended_daily.sort_values(["date", "station"]).reset_index(drop=True)
extended_daily = extended_daily[final_columns]

if extended_daily.duplicated(["station_id", "date"]).any():
    raise ValueError("The extended dataset still contains duplicate station-days.")
if len(extended_daily) != old_rows + added_count:
    raise ValueError("Unexpected final row count after appending and deduplication.")
print(f"Older 2013–2017 rows: {old_rows:,}")
print(f"Newer 2014–2025 rows: {new_rows:,}")
print(f"Overlapping station-days kept from the older file: {overlap_count:,}")
print(f"New station-days added: {added_count:,}")
print(f"Unified rows: {len(extended_daily):,}; columns: {len(extended_daily.columns)}")
print(f"Date span: {extended_daily.date.min()} to {extended_daily.date.max()}")
print(f"Stations: {extended_daily.station.nunique()}; duplicate station-days: {int(extended_daily.duplicated(['station_id', 'date']).sum())}")


Older 2013–2017 rows: 17,492
Newer 2014–2025 rows: 46,680
Overlapping station-days kept from the older file: 12,252
New station-days added: 34,428
Unified rows: 51,920; columns: 34
Date span: 2013-03-01 to 2025-01-04
Stations: 12; duplicate station-days: 0


## 9. Save the unified 2013–2025 dataset to CSV and MySQL

Save a local CSV copy for inspection, and replace the dedicated MySQL table used by the next modeling notebook. The original hourly `all_regions` table is left untouched.


In [10]:
EXTENDED_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
extended_daily.to_csv(EXTENDED_OUTPUT, index=False, encoding="utf-8")
print(f"Saved: {EXTENDED_OUTPUT}")
print(f"Rows: {len(extended_daily):,}; columns: {len(extended_daily.columns)}")
print(f"Dates: {extended_daily.date.min()} to {extended_daily.date.max()}")
print(f"ERA5 weather matched: {int(extended_daily.weather_matched.sum()):,}/{len(extended_daily):,} ({100 * extended_daily.weather_matched.mean():.2f}%)")
print(f"File size: {EXTENDED_OUTPUT.stat().st_size / (1024 ** 2):.2f} MiB")
print(extended_daily.head(5).to_string(index=False))

# Write this daily dataset to its own table; do not overwrite the hourly all_regions table.
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    extended_daily.to_sql(
        DATABASE_TABLE,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1000,
    )
    with engine.connect() as connection:
        database_count = pd.read_sql_query(
            f"SELECT COUNT(*) AS row_count FROM {DATABASE_TABLE}", connection
        ).iloc[0, 0]
    if int(database_count) != len(extended_daily):
        raise RuntimeError(
            f"Database row-count mismatch: expected {len(extended_daily):,}, found {database_count:,}."
        )
    print(f"Loaded air_quality.{DATABASE_TABLE}: {int(database_count):,} rows")
finally:
    engine.dispose()


Saved: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data\daily_pollutant_readings_with_era5_weather_2013-2025.csv
Rows: 51,920; columns: 34
Dates: 2013-03-01 to 2025-01-04
ERA5 weather matched: 51,920/51,920 (100.00%)
File size: 14.70 MiB
      date      station    PM2.5      PM10       SO2       NO2       CO     O3  PM2.5_valid_hours  PM10_valid_hours  SO2_valid_hours  NO2_valid_hours  CO_valid_hours  O3_1h_max station_id weather_station  TEMP_mean  TEMP_range   PRES_mean  DEWP_mean   RH_mean  RAIN_sum  WSPM_mean    wd_sin   wd_cos  temperature_hours  pressure_hours  dewpoint_hours  humidity_hours  rain_hours  wind_speed_hours  wind_direction_hours  weather_matched weather_station_matches_pollutant
2013-03-01 Aotizhongxin 7.125000 10.750000 11.708333 22.583333 0.429167 74.500                 24                24               24               24              24       77.0      1011A    Aotizhongxin   0.508333         9.0 1026.658333 -17.933333 24.29